CNC Turning Operation

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, mean_absolute_percentage_error

In [6]:
df = pd.read_excel("E:/CODE/Python/Project/CNC_turning.xlsx")
df.head(10)

,Run_ID,Experiment,Replica,Tool_ID,Group,Subgroup,Position,Condition,TCond,Machined_length,...,Ra,Rz,Rsk,Rku,RSm,Rt,Fx,Fy,Fz,F
0,1_021_B1_4_a,1,1,21,1,2,a,4,0.0,12,...,0.391,1.855,0.560,2.423,71.4,2.082,49.23,44.46,21.07,69.600499
1,1_021_B1_4_a,1,1,21,1,2,a,4,0.0,12,...,0.359,1.670,0.530,2.229,70.9,1.918,49.23,44.46,21.07,69.600499
2,1_021_B1_4_a,1,1,21,1,2,a,4,0.0,12,...,0.421,1.912,0.634,2.332,71.2,2.062,49.23,44.46,21.07,69.600499
3,1_021_B1_4_a,1,1,21,1,2,a,4,0.0,12,...,0.450,1.961,0.676,2.325,72.8,2.063,49.23,44.46,21.07,69.600499
4,1_021_B1_4_a,1,1,21,1,2,a,4,0.0,12,...,0.360,1.782,0.714,2.636,69.4,1.957,49.23,44.46,21.07,69.600499
5,1_021_B1_4_a,1,1,21,1,2,a,4,0.0,12,...,0.383,1.931,0.580,2.542,67.3,2.039,49.23,44.46,21.07,69.600499
6,1_021_B1_6_b,1,1,21,1,2,b,6,0.0,12,...,1.181,4.853,0.793,2.390,132.5,5.186,81.03,59.21,25.94,103.656011
7,1_021_B1_6_b,1,1,21,1,2,b,6,0.0,12,...,1.139,5.005,0.701,2.492,130.2,5.122,81.03,59.21,25.94,103.656011
8,1_021_B1_6_b,1,1,21,1,2,b,6,0.0,12,...,1.146,5.082,0.847,2.546,129.7,5.249,81.03,59.21,25.94,103.656011
9,1_021_B1_6_b,1,1,21,1,2,b,6,0.0,12,...,1.125,5.044,0.745,2.561,129.3,5.323,81.03,59.21,25.94,103.656011


In [19]:
df[['Run_ID', 'ap', 'vc','f', 'Ra']].head(20)



,Run_ID,ap,vc,f,Ra
0,1_021_B1_4_a,0.25,350,0.07,0.391
1,1_021_B1_4_a,0.25,350,0.07,0.359
2,1_021_B1_4_a,0.25,350,0.07,0.421
3,1_021_B1_4_a,0.25,350,0.07,0.450
4,1_021_B1_4_a,0.25,350,0.07,0.360
5,1_021_B1_4_a,0.25,350,0.07,0.383
6,1_021_B1_6_b,0.25,350,0.13,1.181
7,1_021_B1_6_b,0.25,350,0.13,1.139
8,1_021_B1_6_b,0.25,350,0.13,1.146
9,1_021_B1_6_b,0.25,350,0.13,1.125


In [3]:
df.shape

(612, 27)

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 612 entries, 0 to 611
Data columns (total 27 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Run_ID           612 non-null    object 
 1   Experiment       612 non-null    int64  
 2   Replica          612 non-null    int64  
 3   Tool_ID          612 non-null    int64  
 4   Group            612 non-null    int64  
 5   Subgroup         612 non-null    object 
 6   Position         612 non-null    object 
 7   Condition        612 non-null    int64  
 8   TCond            612 non-null    float64
 9   Machined_length  612 non-null    int64  
 10  Init_diameter    612 non-null    float64
 11  Final_diameter   612 non-null    float64
 12  CTime            612 non-null    object 
 13  R_measurement    612 non-null    int64  
 14  ap               612 non-null    float64
 15  vc               612 non-null    int64  
 16  f                612 non-null    float64
 17  Ra              

In [13]:
df.isnull().sum().sort_values(ascending=True)

Run_ID             0
Experiment         0
Replica            0
Tool_ID            0
Group              0
Subgroup           0
Position           0
Condition          0
TCond              0
Machined_length    0
Init_diameter      0
Final_diameter     0
CTime              0
R_measurement      0
ap                 0
vc                 0
f                  0
Ra                 0
Rz                 0
Rsk                0
Rku                0
RSm                0
Rt                 0
Fx                 0
Fy                 0
Fz                 0
F                  0
dtype: int64

In [15]:
# since we have six Ra with same RunID
# Count rows for each machining run
run_counts = df.groupby("Run_ID").size()
print(run_counts.value_counts())


6    102
Name: count, dtype: int64


we have total 612 column with 6 same run_id, but the Ra was different for each time. So we group by 6 run_id and take the average Ra.

In [21]:
# Average the six Ra measurements for each machining run
df_run = df.groupby("Run_ID", as_index=False).agg({
    "ap": "first",
    "vc": "first",
    "f": "first",
    "Ra": "mean"
})

# Check the new dataset
df_run.head(10)

,Run_ID,ap,vc,f,Ra
0,1_021_B1_1_h,0.25,310,0.07,0.233833
1,1_021_B1_2_g,0.25,310,0.10,0.675000
2,1_021_B1_3_i,0.25,310,0.13,0.829833
3,1_021_B1_4_a,0.25,350,0.07,0.394000
4,1_021_B1_5_c,0.25,350,0.10,0.638500
5,1_021_B1_6_b,0.25,350,0.13,1.160500
6,1_021_B1_7_f,0.25,390,0.07,0.321667
7,1_021_B1_8_d,0.25,390,0.10,0.717333
8,1_021_B1_9_e,0.25,390,0.13,1.102833
9,1_031_B2_10_b,0.50,310,0.07,0.269167


In [22]:
df_run.shape

(102, 5)

In [23]:
df_run.describe()

,ap,vc,f,Ra
count,102.000000,102.000000,102.000000,102.000000
mean,0.450000,350.000000,0.100000,0.682680
std,0.198759,23.880893,0.023631,0.301399
min,0.250000,310.000000,0.070000,0.152000
25%,0.250000,350.000000,0.070000,0.462750
50%,0.500000,350.000000,0.100000,0.630417
75%,0.500000,350.000000,0.130000,0.870250
max,0.800000,390.000000,0.130000,1.748667


Now the dataset as ready for ML

In [27]:
X = df_run.drop(['Run_ID', 'Ra'], axis=1)
y = df_run['Ra']

In [25]:
X.shape

(102, 3)

In [28]:
X_train, X_test, y_train, y_test= train_test_split(X,y, test_size=0.20, random_state=42)


In [29]:
X_train.shape

(81, 3)

In [30]:
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


In [31]:
y_pred= lr_model.predict(X_test)

In [32]:
print("RMSE for Linear Regression Imputaiton: ", np.sqrt(mean_squared_error(y_test, y_pred)))
print("R2 score for Linear Regression Imputation: ", r2_score(y_test, y_pred))
print("MAE for Linear Regression Imputation: ", mean_absolute_error(y_test, y_pred))
print("MAPE for Linear Regression Imputation: ", mean_absolute_percentage_error(y_test, y_pred))

RMSE for Linear Regression Imputaiton:  0.17912600479718208
R2 score for Linear Regression Imputation:  0.5538818193097197
MAE for Linear Regression Imputation:  0.1529661777347852
MAPE for Linear Regression Imputation:  0.32988256110214714


Random Forest

In [33]:
rm_model = RandomForestRegressor()
rm_model.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""squared_error"", ""absolute_error"", ""friedman_mse"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in Poisson deviance to find splits.Training using ""absolute_error"" is significantly slowerthan when using ""squared_error""... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsample

In [34]:
y_pred= rm_model.predict(X_test)

In [35]:
print("RMSE for Random Forest Regression Imputaiton: ", np.sqrt(mean_squared_error(y_test, y_pred)))
print("R2 score for Random Forest Regression Imputation: ", r2_score(y_test, y_pred))
print("MAE for Random Forest Regression Imputation: ", mean_absolute_error(y_test, y_pred))
print("MAPE for Random Forest Regression Imputation: ", mean_absolute_percentage_error(y_test, y_pred))

RMSE for Random Forest Regression Imputaiton:  0.1955460244925012
R2 score for Random Forest Regression Imputation:  0.46834411515022156
MAE for Random Forest Regression Imputation:  0.16327771438856573
MAPE for Random Forest Regression Imputation:  0.3507745325252345


These results indicate that for the present CNC machining dataset, the linear relationship between the cutting parameters and surface roughness is more dominant, and simple linear regression outperformed the random forest imputation approach. However, the moderate R^2 value suggests that surface roughness is not fully explained only by ap, vc and f; other unconsidered factors such as tool condition, vibration or workpiece material properties may also influence the final surface quality.